# Image Classification and Object Localization (PyTorch)

In this lab, you'll build a CNN from scratch to:
- classify the main subject in an image
- localize it by drawing bounding boxes around it.

You'll use the [MNIST](http://yann.lecun.com/exdb/mnist/) dataset to synthesize a custom dataset for the task:
- Place each "digit" image on a black canvas of width 75 x 75 at random locations.
- Calculate the corresponding bounding boxes for those "digits".

The bounding box prediction can be modelled as a "regression" task, which means that the model will predict a numeric value (as opposed to a category).

> This notebook is a PyTorch port of the original TensorFlow lab. The synthetic dataset is a `torch.utils.data.Dataset`, the two-headed network is an `nn.Module` whose `forward` returns both outputs, and Keras' `model.fit` with two losses becomes an explicit training loop that sums the classification and bounding-box losses.

## Imports

In [ ]:
import os
import PIL.Image, PIL.ImageFont, PIL.ImageDraw
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import MNIST
from torchinfo import summary

print("PyTorch version " + torch.__version__)

# Visualization Utilities

These functions are used to draw bounding boxes around the digits.

In [ ]:
#@title Plot Utilities for Bounding Boxes [RUN ME]

im_width = 75
im_height = 75
use_normalized_coordinates = True

def draw_bounding_boxes_on_image_array(image,
                                       boxes,
                                       color=[],
                                       thickness=1,
                                       display_str_list=()):
  '''
  Draws bounding boxes on image (numpy array).

  Args:
    image: a numpy array object.
    boxes: a 2 dimensional numpy array of [N, 4]: (ymin, xmin, ymax, xmax).
           The coordinates are in normalized format between [0, 1].
    color: color to draw bounding box. Default is red.
    thickness: line thickness. Default value is 4.
    display_str_list: a list of strings for each bounding box.

  Returns:
    array -- a new RGBA array with the boxes drawn on it

  Raises:
    ValueError: if boxes is not a [N, 4] array
  '''
  image_pil = PIL.Image.fromarray(image.astype(np.uint8))
  rgbimg = PIL.Image.new("RGBA", image_pil.size)
  rgbimg.paste(image_pil)
  draw_bounding_boxes_on_image(rgbimg, boxes, color, thickness,
                               display_str_list)
  return np.array(rgbimg)


def draw_bounding_boxes_on_image(image,
                                 boxes,
                                 color=[],
                                 thickness=1,
                                 display_str_list=()):
  """Draws bounding boxes on image.
  Args:
    image: a PIL.Image object.
    boxes: a 2 dimensional numpy array of [N, 4]: (ymin, xmin, ymax, xmax).
           The coordinates are in normalized format between [0, 1].
    color: color to draw bounding box. Default is red.
    thickness: line thickness. Default value is 4.
    display_str_list: a list of strings for each bounding box.

  Raises:
    ValueError: if boxes is not a [N, 4] array
  """
  boxes_shape = boxes.shape
  if not boxes_shape:
    return
  if len(boxes_shape) != 2 or boxes_shape[1] != 4:
    raise ValueError('Input must be of size [N, 4]')
  for i in range(boxes_shape[0]):
    draw_bounding_box_on_image(image, boxes[i, 1], boxes[i, 0], boxes[i, 3],
                               boxes[i, 2], color[i], thickness, display_str_list[i])

def draw_bounding_box_on_image(image,
                               ymin,
                               xmin,
                               ymax,
                               xmax,
                               color='red',
                               thickness=1,
                               display_str=None,
                               use_normalized_coordinates=True):
  """Adds a bounding box to an image.
  Bounding box coordinates can be specified in either absolute (pixel) or
  normalized coordinates by setting the use_normalized_coordinates argument.
  Args:
    image: a PIL.Image object.
    ymin: ymin of bounding box.
    xmin: xmin of bounding box.
    ymax: ymax of bounding box.
    xmax: xmax of bounding box.
    color: color to draw bounding box. Default is red.
    thickness: line thickness. Default value is 4.
    display_str_list: string to display in box
    use_normalized_coordinates: If True (default), treat coordinates
      ymin, xmin, ymax, xmax as relative to the image.  Otherwise treat
      coordinates as absolute.
  """
  draw = PIL.ImageDraw.Draw(image)
  im_width, im_height = image.size
  if use_normalized_coordinates:
    (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                  ymin * im_height, ymax * im_height)
  else:
    (left, right, top, bottom) = (xmin, xmax, ymin, ymax)
  draw.line([(left, top), (left, bottom), (right, bottom),
             (right, top), (left, top)], width=thickness, fill=color)

These utilities are used to visualize the data and predictions.

In [ ]:
#@title Visualization Utilities [RUN ME]
"""
This cell contains helper functions used for visualization
and downloads only.

You can skip reading it, as there is very
little PyTorch related code here.
"""

# Matplotlib config
plt.rc('image', cmap='gray')
plt.rc('grid', linewidth=0)
plt.rc('xtick', top=False, bottom=False, labelsize='large')
plt.rc('ytick', left=False, right=False, labelsize='large')
plt.rc('axes', facecolor='F8F8F8', titlesize="large", edgecolor='white')
plt.rc('text', color='a8151a')
plt.rc('figure', facecolor='F0F0F0')# Matplotlib fonts
MATPLOTLIB_FONT_DIR = os.path.join(os.path.dirname(plt.__file__), "mpl-data/fonts/ttf")

# pull a batch from the datasets.
def dataset_to_numpy_util(training_dataset, validation_dataset, N):
  '''
  Pulls a batch from each dataset and returns them as numpy arrays.

  Args:
    training_dataset (Dataset) -- the training split
    validation_dataset (Dataset) -- the validation split
    N (int) -- how many training items to take

  Returns:
    tuple of arrays -- training digits, labels and boxes, then the validation equivalents
  '''
  # get N training digits and the whole validation set as numpy arrays
  training_digits, (training_labels, training_bboxes) = next(iter(DataLoader(training_dataset, batch_size=N, shuffle=True)))
  validation_digits, (validation_labels, validation_bboxes) = next(iter(DataLoader(validation_dataset, batch_size=len(validation_dataset))))

  return (training_digits.numpy(), training_labels.numpy(), training_bboxes.numpy(),
          validation_digits.numpy(), validation_labels.numpy(), validation_bboxes.numpy())

# create digits from local fonts for testing
def create_digits_from_local_fonts(n):
  '''
  Renders digits from matplotlib's bundled fonts, as extra test data.

  Args:
    n (int) -- how many digits to render

  Returns:
    (array, list) -- the rendered digits flattened to (n, 5625), and their labels
  '''
  font_labels = []
  img = PIL.Image.new('LA', (75*n, 75), color = (0,255)) # format 'LA': black in channel 0, alpha in channel 1
  font1 = PIL.ImageFont.truetype(os.path.join(MATPLOTLIB_FONT_DIR, 'DejaVuSansMono-Oblique.ttf'), 25)
  font2 = PIL.ImageFont.truetype(os.path.join(MATPLOTLIB_FONT_DIR, 'STIXGeneral.ttf'), 25)
  d = PIL.ImageDraw.Draw(img)
  for i in range(n):
    font_labels.append(i%10)
    d.text((7+i*75,0 if i<10 else -4), str(i%10), fill=(255,255), font=font1 if i<10 else font2)
  font_digits = np.array(img.getdata(), np.float32)[:,0] / 255.0 # black in channel 0, alpha in channel 1 (discarded)
  font_digits = np.reshape(np.stack(np.split(np.reshape(font_digits, [75, 75*n]), n, axis=1), axis=0), [n, 75*75])
  return font_digits, font_labels


# utility to display a row of digits with their predictions
def display_digits_with_boxes(digits, predictions, labels, pred_bboxes, bboxes, iou, title, iou_threshold=0.6):

  '''
  Displays a row of digits with their predicted and ground truth boxes.

  Args:
    digits (array) -- images, shape (N, 1, 75, 75)
    predictions (array) -- predicted class ids
    labels (array) -- ground truth class ids
    pred_bboxes (array) -- predicted boxes, or an empty array to omit
    bboxes (array) -- ground truth boxes, or an empty array to omit
    iou (array) -- IoU per image, or an empty array to omit
    title (string) -- title for the figure
    iou_threshold (float) -- IoU below which the score is printed in red
  '''
  n = 10

  indexes = np.random.choice(len(predictions), size=n)
  n_digits = digits[indexes]
  n_predictions = predictions[indexes]
  n_labels = labels[indexes]

  n_iou = []
  if len(iou) > 0:
    n_iou = iou[indexes]

  if (len(pred_bboxes) > 0):
    n_pred_bboxes = pred_bboxes[indexes,:]

  if (len(bboxes) > 0):
    n_bboxes = bboxes[indexes,:]


  n_digits = n_digits * 255.0
  n_digits = n_digits.reshape(n, 75, 75)
  fig = plt.figure(figsize=(20, 4))
  plt.title(title)
  plt.yticks([])
  plt.xticks([])

  for i in range(10):
    ax = fig.add_subplot(1, 10, i+1)
    bboxes_to_plot = []
    if (len(pred_bboxes) > i):
      bboxes_to_plot.append(n_pred_bboxes[i])

    if (len(bboxes) > i):
      bboxes_to_plot.append(n_bboxes[i])

    img_to_draw = draw_bounding_boxes_on_image_array(image=n_digits[i], boxes=np.asarray(bboxes_to_plot), color=['red', 'green'], display_str_list=["true", "pred"])
    plt.xlabel(n_predictions[i])
    plt.xticks([])
    plt.yticks([])

    if n_predictions[i] != n_labels[i]:
      ax.xaxis.label.set_color('red')



    plt.imshow(img_to_draw)

    if len(iou) > i :
      color = "black"
      if (n_iou[i][0] < iou_threshold):
        color = "red"
      ax.text(0.2, -0.3, "iou: %s" %(n_iou[i][0]), color=color, transform=ax.transAxes)


# utility to display training and validation curves
def plot_metrics(history, metric_name, title, ylim=5):
  '''
  Plots a training metric and its validation counterpart against the epoch number.

  Args:
    history (dict) -- metric name to list of per-epoch values
    metric_name (string) -- key to plot, for example 'classification_loss'
    title (string) -- title for the figure
    ylim (float) -- upper limit of the y axis
  '''
  plt.title(title)
  plt.ylim(0,ylim)
  plt.plot(history[metric_name],color='blue',label=metric_name)
  plt.plot(history['val_' + metric_name],color='green',label='val_' + metric_name)

## Selecting the Device

### GPU detection

The original lab picked a TensorFlow *distribution strategy* depending on whether a TPU, several GPUs or a single GPU was available. In PyTorch the equivalent for a single machine is simply choosing the `device` the tensors and the model live on:

- `cuda` if an NVIDIA GPU is available,
- `mps` on Apple Silicon Macs,
- otherwise the CPU.

(Multi-GPU training in PyTorch would use `torch.nn.DataParallel` or `DistributedDataParallel`, which is out of scope here.)

In [ ]:
# Detect hardware
if torch.cuda.is_available():
  device = torch.device("cuda")
  print("Running on GPU", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print("Running on Apple MPS")
else:
  device = torch.device("cpu")
  print("Running on CPU")

### Parameters

The batch size per step. (With a single device there is no replica scaling to worry about.)

In [ ]:
BATCH_SIZE = 64

## Loading and Preprocessing the Dataset

Define a `Dataset` class that will pre-process your data:
- `MNISTLocalization.__getitem__`: randomly overlays the "digit" image on top of a larger canvas (this is what `read_image_tfds` did in the TensorFlow version).
- `get_training_dataset`: loads data and splits it to get the training set.
- `get_validation_dataset`: loads and splits the data to get the validation set.

In [ ]:
'''
Transforms each image in dataset by pasting it on a 75x75 canvas at random locations.
'''
class MNISTLocalization(Dataset):

    '''
    MNIST digits pasted at random positions on a 75x75 black canvas.

    Each item is an image together with its class label and the bounding box of the digit.
    Training items are repositioned every time they are fetched, while validation items use
    a fixed position per index so evaluation is repeatable.
    '''
    def __init__(self, train):
        '''
        Loads the MNIST split this dataset pastes onto larger canvases.

        Args:
          train (bool) -- True loads the training split, False the test split
        '''
        # torchvision downloads MNIST to ./data on first use
        self.mnist = MNIST(root="data", train=train, download=True)
        self.train = train

    def __len__(self):
        '''
        Reports how many digits this split holds.

        Returns:
          int -- number of digits in this split
        '''
        return len(self.mnist)

    def __getitem__(self, idx):
        '''
        Pastes digit `idx` at a random position and computes its box.

        Args:
          idx (int) -- index of the digit to fetch

        Returns:
          (tensor, tuple) -- the image (1, 75, 75), and (label, box) where the box is
          [xmin, ymin, xmax, ymax] normalized to [0, 1]
        '''
        digit = self.mnist.data[idx]        # uint8 tensor of shape (28, 28)
        label = int(self.mnist.targets[idx])

        # the training set gets a new random position every time an item is fetched (i.e. every epoch);
        # the validation set uses a fixed position per digit so that evaluation is repeatable
        rng = np.random if self.train else np.random.RandomState(idx)
        xmin = int(rng.randint(0, 48))
        ymin = int(rng.randint(0, 48))

        # paste the digit onto a black 75x75 canvas
        image = torch.zeros((1, 75, 75), dtype=torch.float32)
        image[0, ymin:ymin + 28, xmin:xmin + 28] = digit.float() / 255.0

        xmax = (xmin + 28) / 75
        ymax = (ymin + 28) / 75
        xmin = xmin / 75
        ymin = ymin / 75
        bbox = torch.tensor([xmin, ymin, xmax, ymax], dtype=torch.float32)

        # the class label is an integer (nn.CrossEntropyLoss does not need one-hot targets)
        return image, (label, bbox)

'''
Loads the training split of the dataset. Shuffling and batching are handled by the DataLoader.
'''
def get_training_dataset():
    '''
    Builds the training set and shuffles it into batches.

    Returns:
      (Dataset, DataLoader) -- the training set and shuffled batches over it
    '''
    dataset = MNISTLocalization(train=True)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    return dataset, loader

'''
Loads the validation split of the dataset.
'''
def get_validation_dataset():
    '''
    Builds the validation set and groups it into batches.

    Returns:
      (Dataset, DataLoader) -- the validation set and batches over it, in order
    '''
    dataset = MNISTLocalization(train=False)
    loader = DataLoader(dataset, batch_size=500)  # 10000 items in eval dataset
    return dataset, loader

# instantiate the datasets
training_dataset, training_loader = get_training_dataset()
validation_dataset, validation_loader = get_validation_dataset()

### Visualize Data

In [ ]:
(training_digits, training_labels, training_bboxes,
 validation_digits, validation_labels, validation_bboxes) = dataset_to_numpy_util(training_dataset, validation_dataset, 10)

display_digits_with_boxes(training_digits, training_labels, training_labels, np.array([]), training_bboxes, np.array([]), "training digits and their labels")
display_digits_with_boxes(validation_digits, validation_labels, validation_labels, np.array([]), validation_bboxes, np.array([]), "validation digits and their labels")

## Define the Network

Here, you'll define your custom CNN.
- `feature_extractor`: these convolutional layers extract the features of the image.
- `classifier`:  This define the output layer that predicts among 10 categories (digits 0 through 9)
- `bounding_box_regression`: This defines the output layer that predicts 4 numeric values, which define the coordinates of the bounding box (xmin, ymin, xmax, ymax)
- `LocalizationModel.forward`: This combines the layers for feature extraction, classification and bounding box prediction.  
  - Notice that this is another example of a branching model, because the model splits to produce two kinds of output (a category and set of numbers).  
  - In PyTorch a branching model is just a `forward` method that returns more than one tensor.
- `define_and_compile_model`: choose the optimizer and loss functions.

Note that the classification head returns raw logits (no softmax): `nn.CrossEntropyLoss` applies the softmax internally.

In [ ]:
'''
Feature extractor is the CNN that is made up of convolution and pooling layers.
'''
def feature_extractor():
    '''
    Builds the convolutional stack that extracts features from the canvas.

    Returns:
      nn.Sequential -- three convolution and average pooling stages, producing (N, 64, 7, 7)
    '''
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(),
        nn.AvgPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(),
        nn.AvgPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
        nn.AvgPool2d(2),
    )

'''
dense_layers adds a flatten and dense layer.
This will follow the feature extraction layers
'''
def dense_layers():
    '''
    Builds the shared dense layer that both output heads read from.

    Returns:
      nn.Sequential -- flattening plus a 128 unit ReLU layer
    '''
    # 75 -> 73 -> 36 -> 34 -> 17 -> 15 -> 7, so the features have shape (64, 7, 7)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
    )


'''
Classifier defines the classification output.
This has a fully connected layer producing the 10 class logits (softmax is applied in the loss).
'''
def classifier():
    '''
    Builds the head that predicts which digit is shown.

    Returns:
      nn.Linear -- layer producing the 10 class logits
    '''
    return nn.Linear(128, 10)


'''
This function defines the regression output for bounding box prediction.
Note that we have four outputs corresponding to (xmin, ymin, xmax, ymax)
'''
def bounding_box_regression():
    '''
    Builds the head that predicts where the digit sits.

    Returns:
      nn.Linear -- layer producing 4 box outputs (xmin, ymin, xmax, ymax)
    '''
    return nn.Linear(128, 4)


class LocalizationModel(nn.Module):
    '''
    Branching model that classifies the digit and regresses its bounding box.

    Shared convolution and dense layers feed two heads, so `forward` returns two tensors.
    '''
    def __init__(self):
        '''
        Builds the shared layers and the two output heads.
        '''
        super().__init__()
        self.feature_cnn = feature_extractor()
        self.dense = dense_layers()
        self.classification = classifier()
        self.bounding_box = bounding_box_regression()

    def forward(self, inputs):
        '''
        Extracts shared features, then splits into the classification and box heads.

        Args:
          inputs (tensor) -- batch of images, shape (N, 1, 75, 75)

        Returns:
          (tensor, tensor) -- class logits (N, 10) and predicted boxes (N, 4)
        '''
        feature_cnn = self.feature_cnn(inputs)
        dense_output = self.dense(feature_cnn)

        '''
        The model branches here.
        The dense layer's output gets fed into two branches:
        classification_output and bounding_box_output
        '''
        classification_output = self.classification(dense_output)
        bounding_box_output = self.bounding_box(dense_output)

        return classification_output, bounding_box_output


def define_and_compile_model(device):
    '''
    Creates the model together with its two losses and its optimizer.

    Args:
      device (torch.device) -- device the model is moved to

    Returns:
      (nn.Module, dict, Optimizer) -- the model, a loss per output head, and the Adam optimizer
    '''
    model = LocalizationModel().to(device)

    # one loss per output, exactly like the Keras `loss` dictionary
    losses = {'classification': nn.CrossEntropyLoss(),
              'bounding_box': nn.MSELoss()}
    optimizer = torch.optim.Adam(model.parameters())
    return model, losses, optimizer


model, losses, optimizer = define_and_compile_model(device)

# print model layers
summary(model, input_size=(1, 1, 75, 75), device=device)

### Train and validate the model

Train the model.  
- You can choose the number of epochs depending on the level of performance that you want and the time that you have.
- Each epoch takes a few seconds on a GPU.
- The total loss is the sum of the classification loss and the bounding box loss (this is what Keras does by default when you pass a dictionary of losses).

In [ ]:
EPOCHS = 10 # 45
steps_per_epoch = 60000//BATCH_SIZE  # 60,000 items in this dataset

# the training history, with the same keys Keras would produce
history = {k: [] for k in ['loss', 'classification_loss', 'bounding_box_loss', 'classification_accuracy',
                           'val_loss', 'val_classification_loss', 'val_bounding_box_loss', 'val_classification_accuracy']}


def run_epoch(loader, model, losses, optimizer, device, train, steps=None):
    '''
    Runs one pass over `loader`, training or evaluating both output heads.

    Args:
      loader (DataLoader) -- yields (images, (labels, boxes)) batches
      model (nn.Module) -- the branching classification and localization model
      losses (dict) -- the 'classification' and 'bounding_box' loss functions
      optimizer (Optimizer) -- updates weights; only used when train is True
      device (torch.device) -- device the batches are moved to
      train (bool) -- True updates the weights, False only measures
      steps (int) -- stop after this many batches, or None for the whole loader

    Returns:
      (float, float, float, float) -- total loss, classification loss,
      bounding box loss, and classification accuracy
    '''
    model.train(train)
    sums = {'loss': 0.0, 'classification_loss': 0.0, 'bounding_box_loss': 0.0, 'correct': 0}
    count = 0
    with torch.set_grad_enabled(train):
        for step, (images, (labels, bboxes)) in enumerate(loader):
            if steps is not None and step >= steps:
                break
            images, labels, bboxes = images.to(device), labels.to(device), bboxes.to(device)

            class_logits, pred_bboxes = model(images)
            classification_loss = losses['classification'](class_logits, labels)
            bounding_box_loss = losses['bounding_box'](pred_bboxes, bboxes)
            loss = classification_loss + bounding_box_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            n = len(images)
            sums['loss'] += loss.item() * n
            sums['classification_loss'] += classification_loss.item() * n
            sums['bounding_box_loss'] += bounding_box_loss.item() * n
            sums['correct'] += (class_logits.argmax(1) == labels).sum().item()
            count += n
    return (sums['loss'] / count, sums['classification_loss'] / count,
            sums['bounding_box_loss'] / count, sums['correct'] / count)


for epoch in range(EPOCHS):
    train_metrics = run_epoch(training_loader, model, losses, optimizer, device, train=True, steps=steps_per_epoch)
    val_metrics = run_epoch(validation_loader, model, losses, optimizer, device, train=False)
    for key, value in zip(['loss', 'classification_loss', 'bounding_box_loss', 'classification_accuracy'], train_metrics):
        history[key].append(value)
        history['val_' + key].append(val_metrics[['loss', 'classification_loss', 'bounding_box_loss', 'classification_accuracy'].index(key)])
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_metrics[0]:.4f} - classification_loss: {train_metrics[1]:.4f} "
          f"- bounding_box_loss: {train_metrics[2]:.4f} - classification_accuracy: {train_metrics[3]:.4f} "
          f"- val_loss: {val_metrics[0]:.4f} - val_classification_accuracy: {val_metrics[3]:.4f}")

loss, classification_loss, bounding_box_loss, classification_accuracy = run_epoch(validation_loader, model, losses, optimizer, device, train=False)
print("Validation accuracy: ", classification_accuracy)

In [ ]:
plot_metrics(history, "classification_loss", "Classification Loss")
plot_metrics(history, "bounding_box_loss", "Bounding Box Loss")

## Intersection over union

Calculate the I-O-U metric to evaluate the model's performance.

In [ ]:
def intersection_over_union(pred_box, true_box):
    '''
    Computes the intersection over union of matched box pairs.

    Args:
      pred_box (array) -- predicted boxes, shape (N, 4) as (xmin, ymin, xmax, ymax)
      true_box (array) -- ground truth boxes, same shape and ordering

    Returns:
      array -- IoU per box, shape (N, 1)
    '''
    xmin_pred, ymin_pred, xmax_pred, ymax_pred =  np.split(pred_box, 4, axis = 1)
    xmin_true, ymin_true, xmax_true, ymax_true = np.split(true_box, 4, axis = 1)

    smoothing_factor = 1e-10

    xmin_overlap = np.maximum(xmin_pred, xmin_true)
    xmax_overlap = np.minimum(xmax_pred, xmax_true)
    ymin_overlap = np.maximum(ymin_pred, ymin_true)
    ymax_overlap = np.minimum(ymax_pred, ymax_true)

    pred_box_area = (xmax_pred - xmin_pred) * (ymax_pred - ymin_pred)
    true_box_area = (xmax_true - xmin_true) * (ymax_true - ymin_true)

    overlap_area = np.maximum((xmax_overlap - xmin_overlap), 0)  * np.maximum((ymax_overlap - ymin_overlap), 0)
    union_area = (pred_box_area + true_box_area) - overlap_area

    iou = (overlap_area + smoothing_factor) / (union_area + smoothing_factor)

    return iou

### Visualize predictions
The following code will make predictions and visualize both the classification and the predicted bounding boxes.
- The true bounding box labels will be in green, and the model's predicted bounding boxes are in red.
- The predicted number is shown below the image.

In [ ]:
def predict(model, digits, device, batch_size=64):
    '''
    Runs the model over a numpy array of images.

    Args:
      model (nn.Module) -- the branching classification and localization model
      digits (array) -- images, shape (N, 1, 75, 75)
      device (torch.device) -- device the model runs on
      batch_size (int) -- how many images to push through at a time

    Returns:
      (array, array) -- class probabilities (N, 10) and predicted boxes (N, 4)
    '''
    model.eval()
    probs, boxes = [], []
    with torch.no_grad():
        for i in range(0, len(digits), batch_size):
            batch = torch.from_numpy(digits[i:i + batch_size]).to(device)
            class_logits, pred_bboxes = model(batch)
            probs.append(torch.softmax(class_logits, dim=1).cpu())
            boxes.append(pred_bboxes.cpu())
    return torch.cat(probs).numpy(), torch.cat(boxes).numpy()

# recognize validation digits
predictions = predict(model, validation_digits, device)
predicted_labels = np.argmax(predictions[0], axis=1)

predicted_bboxes = predictions[1]

iou = intersection_over_union(predicted_bboxes, validation_bboxes)

iou_threshold = 0.6

print("Number of predictions where iou > threshold(%s): %s" % (iou_threshold, (iou >= iou_threshold).sum()))
print("Number of predictions where iou < threshold(%s): %s" % (iou_threshold, (iou < iou_threshold).sum()))


display_digits_with_boxes(validation_digits, predicted_labels, validation_labels, predicted_bboxes, validation_bboxes, iou, "True and Predicted values")